# Train Version 2 Outcome And Goal Models

This notebook trains the existing Version 2 architecture with the Version 2 feature upgrade.

Active trained models: CatBoostClassifier for match outcome plus two CatBoostRegressor goal models.

Goal prediction is handled by `catboost_goals_team_1.cbm`, `catboost_goals_team_2.cbm`, and a tuned CatBoost/statistical xG ensemble; prediction then applies goal-scale calibration, Poisson, and score-probability selection.

Version 2 adds leakage-safe recent result form, points form, attacking/defensive form, opponent-strength, and tournament-type features from `data/raw/results.csv`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "core_prediction" / "scripts").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "core_prediction":
    PROJECT_ROOT = cwd.parent
elif cwd.name == "notebooks" and cwd.parent.name == "core_prediction":
    PROJECT_ROOT = cwd.parents[1]
else:
    PROJECT_ROOT = cwd

sys.path.append(str(PROJECT_ROOT / "core_prediction" / "scripts"))

from train_version_2_models import (
    DEFAULT_RESULTS_PATHS,
    DEFAULT_FIFA_PATHS,
    find_existing_path,
    load_results,
    load_latest_fifa,
    build_training_dataset,
    train_models,
    save_training_outputs,
)
from v25_feature_engineering import VERSION_2_5_FEATURE_COLUMNS


## Load Historical Results

In [ ]:
results_path = find_existing_path(DEFAULT_RESULTS_PATHS, "historical results")
fifa_path = find_existing_path(DEFAULT_FIFA_PATHS, "latest FIFA rankings", required=False)

results = load_results(results_path)
latest_fifa = load_latest_fifa(fifa_path)

results.head()


## Build Leakage-Aware Version 2.5 Training Dataset

Keep `allow_latest_rating_features=False` unless you deliberately accept the leakage risk of using current FIFA rankings for old matches.

Rolling form, goal, and opponent-Elo features are computed before each match is added to team history.

In [ ]:
allow_latest_rating_features = False

training_dataset = build_training_dataset(
    results,
    fifa_table=latest_fifa,
    allow_latest_rating_features=allow_latest_rating_features,
)

training_dataset[["date", "team_1", "team_2", "target_outcome", *VERSION_2_5_FEATURE_COLUMNS[:12]]].head()


## Train CatBoost Outcome Classifier

The split remains chronological: older matches train, newer matches validate.

In [ ]:
training_result = train_models(
    training_dataset,
    train_fraction=0.8,
    random_seed=42,
    iterations=500,
)

training_result["metrics"]


## Feature Importance

The saved feature importance should now include Version 2.5 form and tournament-type features where CatBoost finds signal.

In [ ]:
training_result["feature_importance"].head(20)


## Save Training Outputs

Outputs are written to the same existing Version 2 paths.

In [ ]:
save_training_outputs(
    training_dataset,
    training_result,
    allow_latest=allow_latest_rating_features,
)

print("Saved active Version 2 outcome model.")
